# Microbiology and the Hidden Majority of Life Workflow

This notebook scaffold supports the article **Microbiology and the Hidden Majority of Life**. It can be expanded with logistic growth, Monod kinetics, substrate limitation, community recovery, condition scoring, uncertainty screening, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
growth = pd.read_csv(article_dir / 'data' / 'growth_environments.csv')
growth.head()

In [ ]:
def logistic_curve(t, N0, r, K):
    return K / (1 + ((K - N0) / N0) * np.exp(-r * t))

def temp_response(temp, tref=20, q10=2):
    return q10 ** ((temp - tref) / 10)

def ph_response(ph, ph_opt=7, width=1.2):
    return np.exp(-((ph - ph_opt) ** 2) / (2 * width ** 2))

growth['r_eff'] = growth['r0'] * growth['temp'].apply(temp_response) * growth['ph'].apply(ph_response)
growth['abundance_day_48'] = growth.apply(lambda r: logistic_curve(48, r['N0'], r['r_eff'], r['K']), axis=1)
growth[['environment', 'r_eff', 'abundance_day_48']].round(3)

In [ ]:
sites = pd.read_csv(article_dir / 'data' / 'microbial_condition_sites.csv')
sites['microbial_condition_index'] = (
    0.30 * sites['functional_richness'] +
    0.20 * sites['nitrification_potential'] +
    0.20 * sites['denitrification_balance'] +
    0.15 * (1 - sites['pathogen_signal']) +
    0.15 * (1 - sites['organic_overload'])
)
sites.sort_values('microbial_condition_index', ascending=False).round(3)